In [ ]:
import sys
print(sys.executable)

In [ ]:
!pip install -U peft
!nvidia-smi

In [ ]:
!pip install transformers

In [ ]:
!pip install torch==2.11.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
pip install -U torchao

In [ ]:
pip install numpy

In [ ]:
import torch
import numpy as np
import pandas as pd
import os
print("Pytorch version:", torch.__version__)
print("CUDA AVAILABLE:", torch.cuda.is_available())
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Loading model and tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "coded-by-49/LANG_BRIDGE_AI_2.0"

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    token=token
)

model = torch.compile(model)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=token
)


## Importing test data and sampling 2000 lines from the dataset

In [ ]:
from datasets import load_dataset
import random

dataset = load_dataset(
    "coded-by-49/Langbridge_wazobia_data",
    data_files="hau_test_val.jsonl",
    split="train"
)

random.seed(42)
indices = random.sample(range(len(dataset)), 2000)
eval_sample = dataset.select(indices)

## Extracting all 2000 sentence pairs from the pool

In [ ]:
all_native_lang_sentences = [item["hau_Latn"] for item in eval_sample["translation"]]
all_english_sentences     = [item["eng_Latn"] for item in eval_sample["translation"]]

print(f"Pool size: {len(all_native_lang_sentences)} source | {len(all_english_sentences)} reference")

## Tokenized dataset class

In [ ]:
from torch.utils.data import Dataset

class TokenizedDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, index):
        return {key: value[index] for key, value in self.encodings.items()}

## Tokenizing the full 2000-sample pool

In [ ]:
tokenized_native_lang_encodings = tokenizer(
    all_native_lang_sentences,
    truncation=True,
    padding = True, 
    return_tensors = "pt"
)
full_tokenized_dataset = TokenizedDataset(tokenized_native_lang_encodings)
print(f"Full tokenized dataset size: {len(full_tokenized_dataset)}")

## Data collator

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

## ⚙️ Evaluation Configuration
This is the **only cell you need to change** between runs.
- `EVAL_START` and `EVAL_END` define the total window from the 2000-sample pool
- `CHUNK_SIZE` must stay at 100 (GPU memory limit)
- `LANG_PAIR` is used to name the output file

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
EVAL_START  = 600
EVAL_END    = 1600
CHUNK_SIZE  = 100
LANG_PAIR   = "hausa_eng"
OUTPUT_PATH = f"l1_{LANG_PAIR}_comet_eval_data_{EVAL_START}_{EVAL_END}.json"
# ────────────────────────────────────────────────────────────────────────────

assert EVAL_END <= 2000, "EVAL_END cannot exceed the 2000-sample pool size"
assert EVAL_START < EVAL_END, "EVAL_START must be less than EVAL_END"
assert CHUNK_SIZE <= 100, "CHUNK_SIZE above 100 risks GPU OOM"

total_samples = EVAL_END - EVAL_START
num_chunks = (total_samples + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"Evaluating indices [{EVAL_START}, {EVAL_END}) → {total_samples} samples across {num_chunks} chunks")
print(f"Output → {OUTPUT_PATH}")

## 🔁 Automated Chunked Inference Loop
Iterates over the evaluation window in steps of `CHUNK_SIZE`.
Each chunk appends `{src, mt, ref}` dicts to `comet_eval_data`.
No manual re-running or copy-pasting needed.

In [ ]:
import gc
import json
from torch.utils.data import DataLoader, Subset

local_lang_translated_to = "eng_Latn"
comet_eval_data = []

model.eval()

for chunk_idx, chunk_start in enumerate(range(EVAL_START, EVAL_END, CHUNK_SIZE)):
    chunk_end = min(chunk_start + CHUNK_SIZE, EVAL_END)
    print(f"\n── Chunk {chunk_idx + 1}/{num_chunks}: indices [{chunk_start}, {chunk_end}) ──")

    chunk_src_sentences = all_native_lang_sentences[chunk_start:chunk_end]
    chunk_ref_sentences = all_english_sentences[chunk_start:chunk_end]

    chunk_dataset  = Subset(full_tokenized_dataset, range(chunk_start, chunk_end))
    chunk_loader   = DataLoader(
        dataset=chunk_dataset,
        batch_size=8,
        shuffle=False,
        num_workers=4,
        collate_fn=data_collator
    )

    # ── Inference ────────────────────────────────────────────────────────────
    chunk_output_ids = []

    with torch.no_grad():
        for batch in chunk_loader:
            input_ids      = batch["input_ids"].to(model.device)
            attention_mask = batch["attention_mask"].to(model.device)

            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=256,
                num_beams=4,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(local_lang_translated_to),
                no_repeat_ngram_size=3
            )

            chunk_output_ids.extend(generated_ids)

            del input_ids, attention_mask, generated_ids
            gc.collect()
            torch.cuda.empty_cache()

    # ── Decode ───────────────────────────────────────────────────────────────
    chunk_mt_texts = tokenizer.batch_decode(chunk_output_ids, skip_special_tokens=True)

    # ── Concordance check ────────────────────────────────────────────────────
    assert len(chunk_src_sentences) == len(chunk_mt_texts) == len(chunk_ref_sentences), (
        f"Chunk {chunk_idx + 1}: length mismatch — "
        f"src={len(chunk_src_sentences)}, mt={len(chunk_mt_texts)}, ref={len(chunk_ref_sentences)}"
    )

    for src, mt, ref in zip(chunk_src_sentences, chunk_mt_texts, chunk_ref_sentences):
        comet_eval_data.append({"src": src, "mt": mt, "ref": ref})

    print(f"   ✓ Accumulated {len(comet_eval_data)} total samples so far")

    del chunk_output_ids, chunk_mt_texts, chunk_dataset, chunk_loader
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ Inference complete — {len(comet_eval_data)} samples accumulated")

## 💾 Writing AfriCOMET evaluation data to file
Writes the full accumulator as a JSON array, then clears it from memory.

In [ ]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(comet_eval_data, f, ensure_ascii=False, indent=4)

print(f"✅ Saved {len(comet_eval_data)} samples → {OUTPUT_PATH}")

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    verification = json.load(f)

print(f"✅ Verification: {len(verification)} records readable from file")
print(f"   First record: {verification[0]}")
print(f"   Last  record: {verification[-1]}")

# Clear accumulator from memory
del comet_eval_data, verification
gc.collect()
print("\n🧹 Accumulator cleared from memory")